# Tutorial 2: Basis Set Study

**Comparing accuracy vs. computational cost**

## Learning Objectives
- Understand what a basis set is and why it matters
- Compare STO-3G, 6-31G, and cc-pVDZ on the same molecule
- Interpret convergence patterns in total energy
- Balance accuracy with computational cost

## Time: 30–35 minutes

---

### Background

A **basis set** is the mathematical language we use to describe molecular orbitals.
Larger basis sets give more accurate results but take longer to compute.

| Basis set | Size | Speed | Accuracy |
|-----------|------|-------|----------|
| STO-3G    | Minimal  | Very fast | Low  |
| 6-31G     | Split-valence | Fast | Moderate |
| cc-pVDZ   | Double-zeta | Slower | High |

We will run **RHF** (Restricted Hartree-Fock) on water (H₂O) with each basis set
and compare the total energies.


In [ ]:
# QuantUI-local — in-session calculation imports
from quantui import (
    Molecule,
    parse_xyz_input,
)

try:
    from quantui import SessionResult, run_in_session
    PYSCF_AVAILABLE = True
except (ImportError, AttributeError):
    PYSCF_AVAILABLE = False
    print("PySCF not available — calculations will not run.")
    print("PySCF requires Linux, macOS, or WSL.")


In [ ]:
# Define the water molecule
water_xyz = """O  0.000  0.000  0.000
H  0.757  0.587  0.000
H -0.757  0.587  0.000"""

atoms, coords = parse_xyz_input(water_xyz)
molecule = Molecule(atoms, coords, charge=0, multiplicity=1)
print(f"Molecule: {molecule.get_formula()} ready")


In [ ]:
# Run RHF with three basis sets in-session
import threading

basis_sets = ["STO-3G", "6-31G", "cc-pVDZ"]
basis_results = {}  # basis_name -> SessionResult

if not PYSCF_AVAILABLE:
    print("Skipping calculations — PySCF not available.")
else:
    def _run_one(basis):
        try:
            print(f"Starting RHF/{basis} ...")
            result = run_in_session(molecule=molecule, method="RHF", basis=basis)
            basis_results[basis] = result
            print(f"  {basis}: {result.energy:.8f} Ha")
        except Exception as exc:
            print(f"  ERROR for {basis}: {exc}")

    threads = [threading.Thread(target=_run_one, args=(b,), daemon=True) for b in basis_sets]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    print("\nAll calculations complete.")


In [ ]:
# Compare energies (all values in Hartree and kcal/mol)
if basis_results:
    print(f"{'Basis set':12s}  {'Energy (Ha)':>15s}  {'vs cc-pVDZ (kcal/mol)':>22s}")
    print("-" * 54)
    reference = basis_results.get("cc-pVDZ")
    for basis in basis_sets:
        if basis not in basis_results:
            print(f"{basis:12s}  {'(failed)':>15s}")
            continue
        energy = basis_results[basis].energy
        if reference is not None:
            diff_kcal = (energy - reference.energy) * 627.5095
            print(f"{basis:12s}  {energy:>15.8f}  {diff_kcal:>+22.3f}")
        else:
            print(f"{basis:12s}  {energy:>15.8f}  {'(no reference)':>22s}")
else:
    print("No results to compare. Run the cell above first.")


## Complete!

**Key Insights:**
- **STO-3G**: Fastest, but the largest error (~30–100 kcal/mol from the reference)
- **6-31G**: Good balance — much more accurate than STO-3G, still quick
- **cc-pVDZ**: Our reference; higher accuracy, takes slightly longer

**Questions to explore:**
1. Which basis set gives the lowest (most negative) energy? Why?
2. How does computation time scale with basis set size?
3. For a quick check, is STO-3G acceptable? For publication quality?

**Next:** [Tutorial 3: Multiplicity and Radicals](03_multiplicity_radicals.ipynb)
